# Phase 2 — LSTM Volatility Forecasting

This notebook trains a global LSTM on AAPL and compares it against the Phase 1 baselines on identical walk-forward splits.

**Model:** `VolatilityLSTM` — single LSTM layer (hidden=64) + linear head  
**Features:** `log_returns`, `rv_5d`, `rv_20d`, `rv_60d` (scaled per fold, train only)  
**Evaluation:** 5 expanding-window folds, horizon=5, min_train=252 days

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader

from src.data.ingest import DEFAULT_RAW_DIR
from src.eval.evaluate import evaluate_baselines, evaluate_lstm
from src.eval.walk_forward import walk_forward_splits
from src.features.dataset import VolatilityDataset
from src.features.realized_vol import build_features, log_returns
from src.models.lstm import VolatilityLSTM
from src.models.train import predict, train_model

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 4)

## 1. Load data

In [2]:
df = pd.read_parquet(DEFAULT_RAW_DIR / 'AAPL.parquet')
prices = df['Adj Close']
returns = log_returns(prices).dropna()

print(f"{len(prices)} trading days: {prices.index[0].date()} → {prices.index[-1].date()}")

11476 trading days: 1980-12-12 → 2026-06-26


## 2. Feature pipeline

`build_features` computes four features from prices and a forward-looking target, then drops all NaN rows.

In [3]:
features = build_features(prices, horizon=5)

print(f"Feature DataFrame: {features.shape[0]} rows × {features.shape[1]} cols")
print(f"Date range: {features.index[0].date()} → {features.index[-1].date()}")
features.head(3)

Feature DataFrame: 11411 rows × 5 cols
Date range: 1981-03-11 → 2026-06-18


,log_returns,rv_5d,rv_20d,rv_60d,rv_target
date,,,,,
1981-03-11 00:00:00-05:00,-0.039663,0.494746,0.585933,0.606687,0.430988
1981-03-12 00:00:00-05:00,0.039663,0.723892,0.607257,0.604721,0.528474
1981-03-13 00:00:00-05:00,-0.011173,0.721637,0.607342,0.585554,0.453745


## 3. Walk-forward evaluation

Both models are evaluated on **identical** 5-fold expanding-window splits. The LSTM scales features on the training slice only — the scaler is re-fit fresh each fold.

In [4]:
baseline_results = evaluate_baselines(returns, horizon=5, n_splits=5, min_train_size=252)
baseline_results.round(4)

,rmse,mae,qlike
naive,0.2570,0.1698,1.6150
ewma,0.2195,0.1508,0.5792
garch,0.2441,0.1892,0.6962


In [ ]:
# ~2-3 minutes: trains a fresh model on each of the 5 folds
from src.config import load_config

cfg = load_config()
lstm_results = evaluate_lstm(
    prices,
    cfg
)
lstm_results.round(4)

,rmse,mae,qlike
lstm,0.2061,0.14,0.6026


## 4. Results comparison

In [6]:
all_results = pd.concat([baseline_results, lstm_results])
all_results.round(4)

,rmse,mae,qlike
naive,0.2570,0.1698,1.6150
ewma,0.2195,0.1508,0.5792
garch,0.2441,0.1892,0.6962
lstm,0.2061,0.1400,0.6026


## 5. Predictions vs actuals — last fold

Visualise LSTM forecasts against realised volatility on the most recent test fold. EWMA is included as the baseline to beat.

In [ ]:
FEATURE_COLS = ["log_returns", "rv_5d", "rv_20d", "rv_60d"]
TARGET_COL = "rv_target"

# Reuse the same config that drove evaluate_lstm above — no hard-coded hyperparameters.
splits = walk_forward_splits(
    features.index,
    n_splits=cfg.eval.n_splits,
    horizon=cfg.data.horizon,
    min_train_size=cfg.eval.min_train_size,
)
train_dates, test_dates = splits[-1]

train_df = features.loc[train_dates]
test_df = features.loc[test_dates].copy()

n_val = int(len(train_df) * cfg.eval.val_frac)
inner_train_df = train_df.iloc[:-n_val].copy()
val_df = train_df.iloc[-n_val:].copy()

scaler = StandardScaler()
inner_train_df[FEATURE_COLS] = scaler.fit_transform(inner_train_df[FEATURE_COLS])
val_df[FEATURE_COLS] = scaler.transform(val_df[FEATURE_COLS])
test_df[FEATURE_COLS] = scaler.transform(test_df[FEATURE_COLS])

train_loader = DataLoader(
    VolatilityDataset(inner_train_df, FEATURE_COLS, TARGET_COL, cfg.model.seq_len),
    batch_size=cfg.train.batch_size, shuffle=True,
)
val_loader = DataLoader(
    VolatilityDataset(val_df, FEATURE_COLS, TARGET_COL, cfg.model.seq_len),
    batch_size=cfg.train.batch_size, shuffle=False,
)

model = VolatilityLSTM(
    input_size=len(FEATURE_COLS),
    hidden_size=cfg.model.hidden_size,
    num_layers=cfg.model.num_layers,
    dropout=cfg.model.dropout,
)
model = train_model(
    model, train_loader, val_loader,
    epochs=cfg.train.epochs, lr=cfg.train.lr, patience=cfg.train.patience,
)

lstm_preds = predict(model, test_df, FEATURE_COLS, TARGET_COL, cfg.model.seq_len)
actuals = features.loc[test_dates, TARGET_COL].dropna()
common = lstm_preds.index.intersection(actuals.index)

from src.models.baselines import ewma_forecast
ewma_preds = ewma_forecast(returns, span=32).loc[common]

fig, ax = plt.subplots()
actuals.loc[common].plot(ax=ax, label='Realized vol (actual)', color='black', linewidth=1)
lstm_preds.loc[common].plot(ax=ax, label='LSTM forecast', color='steelblue', linewidth=1.5)
ewma_preds.plot(ax=ax, label='EWMA forecast', color='tomato', linewidth=1.5, linestyle='--')
ax.set_title('AAPL — LSTM vs EWMA vs realized volatility (last fold)')
ax.set_ylabel('Annualized volatility')
ax.legend()
plt.tight_layout()

## 6. Results discussion

**LSTM beats EWMA on RMSE (~6%) and MAE (~7%), but is slightly worse on QLIKE (0.60 vs 0.58).**

### What this means

- **RMSE and MAE** measure average forecast error in volatility units. The LSTM's lower values mean its point forecasts are closer to the realised outcome than EWMA's — the neural network is learning a richer mapping from the 30-day feature sequence than the exponential average can capture.

- **QLIKE** is a loss function designed specifically for volatility models. It penalises under-prediction more heavily than over-prediction (because under-estimating risk is more dangerous). EWMA's slight edge here reflects a known property: exponential smoothing is well-calibrated for short-horizon equity vol and rarely under-predicts badly.

- **GARCH is beaten on all three metrics** — meaningful because GARCH is a parametric model purpose-built for volatility, and it still loses to both EWMA and the LSTM on this dataset and horizon.

### Why this result is credible

A large LSTM win over EWMA would be a red flag for data leakage. A modest improvement on two of three metrics — with EWMA holding its ground on QLIKE — is exactly the honest, expected outcome described in the project build plan. The LSTM adds genuine value on mean error metrics while EWMA remains the QLIKE champion at this horizon.